# **Personalized Information Retrieval: Course Project**
By Chiara Bosi, Leonardo Castelli, Francesca Chite

**Download of the datasets**

In [ ]:
! gdown 1HhgXzyEpsZNcenU9XhJuOYyDUKEzUse4

Downloading...
From (original): https://drive.google.com/uc?id=1HhgXzyEpsZNcenU9XhJuOYyDUKEzUse4
From (redirected): https://drive.google.com/uc?id=1HhgXzyEpsZNcenU9XhJuOYyDUKEzUse4&confirm=t&uuid=60fe9fca-44e6-4c8e-bd53-ff32144959f5
To: /content/pir_data.zip
100% 3.30G/3.30G [00:29<00:00, 111MB/s]


In [ ]:
! gdown 18m-XzCdBPJHDn2rcgDAnUi4MVoqERmfJ

Downloading...
From (original): https://drive.google.com/uc?id=18m-XzCdBPJHDn2rcgDAnUi4MVoqERmfJ
From (redirected): https://drive.google.com/uc?id=18m-XzCdBPJHDn2rcgDAnUi4MVoqERmfJ&confirm=t&uuid=e8139085-f697-48a9-9299-d38f67c2024c
To: /content/answer_retrieval.zip
100% 26.9M/26.9M [00:00<00:00, 123MB/s]


In [ ]:
!unzip answer_retrieval.zip

Archive:  answer_retrieval.zip
   creating: answer_retrieval/
  inflating: answer_retrieval/subset_answers.json  
   creating: answer_retrieval/test/
  inflating: answer_retrieval/test/subset_test.jsonl  
  inflating: answer_retrieval/test/test_qrels.json  
   creating: answer_retrieval/train/
  inflating: answer_retrieval/train/subset_train.jsonl  
  inflating: answer_retrieval/train/train_qrels.json  
   creating: answer_retrieval/val/
  inflating: answer_retrieval/val/subset_val.jsonl  
  inflating: answer_retrieval/val/val_qrels.json  


In [ ]:
import pandas as pd
import os
import re
import numpy as np

**Data preprocessing for validation and test sets**

In [ ]:
def clean_text(text):

  text = text.lower()
  text = re.sub(r'http[s]?://\S+', '', text)    # Remove URLs
  text = re.sub(r'<.*?>', '', text)             # Remove HTML tags
  text = re.sub(r"\\", "", text)                # Remove backslashes
  text = re.sub(r"[^a-zA-Z0-9\s]", "", text)    # Remove special characters and punctuation
  text = re.sub(r'\d+', '', text)               # Remove numbers
  text = re.sub(r"\s+", " ", text)              # Remove extra whitespace
  text = text.strip()

  return text

In [ ]:
# Load the corpus: subset_answers
# renamed columns: docno (doc number), text (textual content of the answer)

corpus_df = pd.read_json('answer_retrieval/subset_answers.json', orient='index')
corpus_df = corpus_df.reset_index()
corpus_df.columns = ['docno', 'text']

# clean the text

corpus_df['text'] = corpus_df['text'].apply(clean_text)

# corpus_df.head(2)

In [ ]:
# Load and preprocess validation and test queries

val_queries = pd.read_json('answer_retrieval/val/subset_val.jsonl', lines=True)
test_queries = pd.read_json('answer_retrieval/test/subset_test.jsonl', lines=True)

# renamed columns of test and val_queries: quid (query ID), query (query text), user_id

val_queries = val_queries[['id','text','user_id']]
val_queries.columns = ['qid', 'query', 'user_id']

val_queries['query'] = val_queries['query'].apply(clean_text)

test_queries = test_queries[['id','text','user_id']]
test_queries.columns = ['qid', 'query', 'user_id']
test_queries['query'] = test_queries['query'].apply(clean_text)

# test_queries.head(2)

<ipython-input-7-c02a0ed1f8e5>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_queries['query'] = val_queries['query'].apply(clean_text)


Loading the data

In [ ]:
# Load and prepare ground truth data (qrels) for evaluating IR performance
# Add to test and val_qrels a 'label' column with values set to 1 to indicate relevance to the query

val_qrels = pd.read_json('answer_retrieval/val/val_qrels.json', orient='index').reset_index()
val_qrels.columns = ['qid', 'docno']
val_qrels['label'] = 1

test_qrels = pd.read_json('answer_retrieval/test/test_qrels.json', orient='index').reset_index()
test_qrels.columns = ['qid', 'docno']
test_qrels['label'] = 1

In [ ]:
display(corpus_df.head(2))
display(val_queries.head(2))
display(val_qrels.head(2))

,docno,text
0,writers_2000,i like that youre playing with different viewp...
1,writers_2018,your writing style is streamofconsciousness wh...


,qid,query,user_id
0,academia_143743,on answering a question that no one has asked ...,1582241
1,academia_148899,how much domain expertise and network does a s...,935589


,qid,docno,label
0,academia_143743,academia_143753,1
1,academia_148899,academia_148936,1


In [ ]:
# REDUCED DATASETS for faster computations

# corpus_df = corpus_df.head(500)
# val_qrels = val_qrels.head(500)
# test_qrels = test_qrels.head(500)
val_queries = val_queries.head(50)
test_queries = test_queries.head(50)


In [ ]:
# Download tools to apply lemmatization, tokenization and stopword removal

from nltk.stem import WordNetLemmatizer
from nltk.tokenize import WordPunctTokenizer
import nltk

nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
# Apply stopword removal and lemmatization
# note: sparsity reduction

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

nltk.download('wordnet')
nltk.download('stopwords')


wordnet_lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def lemmatizer_with_stopwords(text):
    # Tokenize the text into individual words
    words = text.split()
    # Remove stop words and lemmatize the remaining words
    lemm_text = " ".join([wordnet_lemmatizer.lemmatize(word) for word in words if word not in stop_words])
    return lemm_text


val_queries['query'] = val_queries['query'].apply(lemmatizer_with_stopwords)
test_queries['query'] = test_queries['query'].apply(lemmatizer_with_stopwords)
corpus_df['text'] = corpus_df['text'].apply(lemmatizer_with_stopwords)
# display(val_queries.head(5))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


The main goal of IR is to retrieve all relevant documents with respect to users’ queries, while minimizing the number of non-relevant documents; the initial step to have a more structured representation of the data, is to clean both the corpus and queries from redundancies and noise.

# Exploring TF-IDF

Manual approach for basic IR

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Set the vectorizer
tv = TfidfVectorizer(min_df=0., max_df=1., use_idf=True)

# Transform the data to vectors
tv_matrix = tv.fit_transform(corpus_df.text[:500])
tv_matrix = tv_matrix.toarray()

# Create a pandas DataFrame to display the TF-IDF values
vocab = tv.get_feature_names_out()
pd.DataFrame(np.round(tv_matrix, 2), columns=vocab)

,aaa,abandoned,abercrombie,ability,abkhazia,able,aboard,abomination,aboutfor,aboutit,...,zero,zheng,zinc,zipper,zombie,zone,zoo,zoonotic,zurich,zvv
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
496,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
497,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
498,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# doc similarity with a given text
# Retrieve n most similar documents for each query

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# vectorizer = TfidfVectorizer(ngram_range=(1,3))
vectorizer = TfidfVectorizer(min_df=0., max_df=1., use_idf=True)
docs_tfidf = vectorizer.fit_transform(corpus_df.text)

def get_tf_idf_query_similarity(vectorizer, docs_tfidf, query):

    query_tfidf = vectorizer.transform([query])

    cosineSimilarities = cosine_similarity(query_tfidf, docs_tfidf).flatten()
    return cosineSimilarities

# similarity_list = get_tf_idf_query_similarity(vectorizer, docs_tfidf, val_queries['query'][0])

def get_top_n_documents(vectorizer, docs_tfidf, query_row, n=10):
    query = query_row['query']  # Extract query text from the row
    similarity_scores = get_tf_idf_query_similarity(vectorizer, docs_tfidf, query)
    top_n_indices = np.argsort(similarity_scores)[::-1][:n]  # Get indices of top N documents
    top_n_documents = corpus_df.iloc[top_n_indices].copy()  # Create a copy to avoid modifying original

    # Add query information as new columns
    for col in query_row.index:
        top_n_documents[col] = query_row[col]

    # similarity rank column
    top_n_documents['similarity'] = range(1, len(top_n_documents) + 1)


    return top_n_documents

# Example usage for the first query (now passing the entire row)
top_docs_df = get_top_n_documents(vectorizer, docs_tfidf, val_queries.iloc[0])
display(top_docs_df.head(5))

top_docs_df = get_top_n_documents(vectorizer, docs_tfidf, test_queries.iloc[0])
display(top_docs_df.head(5))

,docno,text,qid,query,user_id,similarity
10828,academia_29156,interesting question project feel stale studen...,academia_143743,answering question one asked historical resear...,1582241,1
4687,linguistics_3011,project may looking,academia_143743,answering question one asked historical resear...,1582241,2
253,workplace_13766,general completed project part course youd oft...,academia_143743,answering question one asked historical resear...,1582241,3
297,workplace_102543,general answering question addressed someone e...,academia_143743,answering question one asked historical resear...,1582241,4
4624,literature_782,graphic novel subcategory comic book turn subs...,academia_143743,answering question one asked historical resear...,1582241,5


,docno,text,qid,query,user_id,similarity
10928,academia_185179,georgetown university named village georgetown...,academia_185177,george georgetown university named george geor...,1532620,1
10889,academia_73203,applicant particular school may widely dispers...,academia_185177,george georgetown university named george geor...,1532620,2
10887,academia_68906,question based wrong premise funding specifica...,academia_185177,george georgetown university named george geor...,1532620,3
10800,academia_16379,one side effect university famous sport team m...,academia_185177,george georgetown university named george geor...,1532620,4
10613,anime_50580,yes form make appearence named dragon ball her...,academia_185177,george georgetown university named george geor...,1532620,5


In [ ]:
# Iterate through all queries
all_top_docs = []
for i in range(test_queries.shape[0]):
    top_docs = get_top_n_documents(vectorizer, docs_tfidf, test_queries.iloc[i])
    all_top_docs.append(top_docs)

# Concatenate all results into a single DataFrame
all_top_docs_df = pd.concat(all_top_docs, ignore_index=True)
all_top_docs_df = all_top_docs_df[['qid', 'query', 'user_id', 'text', 'docno', 'similarity']]
tf_idf_retrieval_result = all_top_docs_df.copy()
all_top_docs_df.head(15)

,qid,query,user_id,text,docno,similarity
0,academia_185177,george georgetown university named george geor...,1532620,georgetown university named village georgetown...,academia_185179,1
1,academia_185177,george georgetown university named george geor...,1532620,applicant particular school may widely dispers...,academia_73203,2
2,academia_185177,george georgetown university named george geor...,1532620,question based wrong premise funding specifica...,academia_68906,3
3,academia_185177,george georgetown university named george geor...,1532620,one side effect university famous sport team m...,academia_16379,4
4,academia_185177,george georgetown university named george geor...,1532620,yes form make appearence named dragon ball her...,anime_50580,5
5,academia_185177,george georgetown university named george geor...,1532620,think may accident earliest reference found en...,history_20453,6
6,academia_185177,george georgetown university named george geor...,1532620,university enforced corporate identity since d...,academia_10499,7
7,academia_185177,george georgetown university named george geor...,1532620,university experience hold trial order reach d...,academia_27056,8
8,academia_185177,george georgetown university named george geor...,1532620,focusing uk start university seem attempting p...,academia_69225,9
9,academia_185177,george georgetown university named george geor...,1532620,order appearancegerdes ian gerdes level design...,gaming_109142,10


TF-IDF, which relates the importance of a word to its frequency and relevance in the corpus, is applied: the aim here is to generate a numerical vector (storing TF-IDF scores) for each document.
Finally, the cosine similarity between the representations of queries and documents is computed and, among the final ranking, the top ten results are selected.

# More advanced IR

On the same subset of queries, different embeddings strategies were applied, in order to determine the optimal one.

In [ ]:
# corpus: subset_answers
# renamed columns: docno (doc number), text (textual content of the answer)

corpus_df = pd.read_json('answer_retrieval/subset_answers.json', orient='index')
corpus_df = corpus_df.reset_index()
corpus_df.columns = ['docno', 'text']

corpus_df['text'] = corpus_df['text'].apply(clean_text)

# corpus_df.head(2)

In [ ]:
val_queries = pd.read_json('answer_retrieval/val/subset_val.jsonl', lines=True)
test_queries = pd.read_json('answer_retrieval/test/subset_test.jsonl', lines=True)

val_queries = val_queries[['id','text','user_id']]
val_queries.columns = ['qid', 'query', 'user_id']

val_queries['query'] = val_queries['query'].apply(clean_text)

test_queries = test_queries[['id','text','user_id']]
test_queries.columns = ['qid', 'query', 'user_id']
test_queries['query'] = test_queries['query'].apply(clean_text)

# test_queries.head(2)

<ipython-input-17-bae3b673c7eb>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_queries['query'] = val_queries['query'].apply(clean_text)


In [ ]:
val_qrels = pd.read_json('answer_retrieval/val/val_qrels.json', orient='index').reset_index()
val_qrels.columns = ['qid', 'docno']
val_qrels['label'] = 1

test_qrels = pd.read_json('answer_retrieval/test/test_qrels.json', orient='index').reset_index()
test_qrels.columns = ['qid', 'docno']
test_qrels['label'] = 1

In [ ]:
# REDUCED DATASETS for faster computations

# corpus_df = corpus_df.head(500)
# val_qrels = val_qrels.head(500)
# test_qrels = test_qrels.head(500)
val_queries = val_queries.head(50)
test_queries = test_queries.head(50)

In [ ]:
# PyTerrier download
!pip install --upgrade -q python-terrier==0.12.1

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.9/347.9 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.9/287.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.3 MB/s eta 0:00:00


In [ ]:
# PyTerrier initialization
import pyterrier as pt
if not pt.started():
    pt.init(tqdm='notebook')


<ipython-input-21-afd1dbbb70c9>:3: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...
Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...
Done


Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
<ipython-input-21-afd1dbbb70c9>:4: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
The following code will have the same effect:
pt.utils.set_tqdm('notebook')
pt.java.init() # optional, forces java initialisation
  pt.init(tqdm='notebook')


Indexing

In [ ]:
# indexing corpus_df

pqa_index_path = './se-pqa'

if not os.path.exists(pqa_index_path + '/data.properties'):
  indexer = pt.index.IterDictIndexer(pqa_index_path, overwrite=True)
  index_ref = indexer.index(
      corpus_df.to_dict(orient='records'),
      fields=({'text':2096}),
      meta=({'docno': 20, 'text': 20096})
  )

else:
  index_ref = pt.IndexRef.of(pqa_index_path + '/data.properties')

index = pt.IndexFactory.of(index_ref)

11:41:56.876 [ForkJoinPool-1-worker-3] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (english_170632) - further warnings are suppressed
11:42:08.446 [ForkJoinPool-1-worker-3] WARN org.terrier.structures.indexing.Indexer -- Indexed 1 empty documents


In [ ]:
print(index.getCollectionStatistics().toString())

Number of documents: 10929
Number of terms: 106275
Number of postings: 824758
Number of fields: 1
Number of tokens: 1234240
Field names: [text]
Positions:   false



In [ ]:
!pip install -q sentence_transformers ipdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.7 MB/s eta 0:00:00


In [ ]:
# Transform the queries into embeddings using Crossencoders and Biencoders

from sentence_transformers import CrossEncoder, SentenceTransformer
crossmodel = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
biencoder_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Encoders**

Crossencoders and biencoders

In [ ]:
# Apply Crossencoders (compare query-document pairs) and Bi-encoders (encode queries and documents separately)

def _crossencoder_apply(df, column='text'):
  return crossmodel.predict(list(zip(df['query'].values, df[column].values)))

from functools import partial
crossencoder_apply_title = partial(_crossencoder_apply, column='text')
cross_encT = pt.apply.doc_score(crossencoder_apply_title, batch_size=64)

from sentence_transformers.util import cos_sim
def _biencoder_apply(df):
  query_embs = biencoder_model.encode(df['query'].values)
  doc_embs = biencoder_model.encode(df['text'].values)
  scores = cos_sim(query_embs, doc_embs)
  return scores[0]

bi_encT = pt.apply.doc_score(_biencoder_apply, batch_size=64)

In [ ]:
# Complete pipeline
# model comparison on validation data

br = pt.BatchRetrieve(index, wmodel='TF_IDF') % 100 # rerank only the top-100, TF_IDF, BM25

bi_pipeline = br >> pt.text.get_text(index, 'text') >> bi_encT
cross_pipeline = br >> pt.text.get_text(index, 'text') >> cross_encT

normalized_br = br >> pt.pipelines.PerQueryMaxMinScoreTransformer()

normalized_bi_pipeline = bi_pipeline >> pt.pipelines.PerQueryMaxMinScoreTransformer()
normalized_cross_pipeline = cross_pipeline >> pt.pipelines.PerQueryMaxMinScoreTransformer()

bi_sum_pipeline = .5*normalized_bi_pipeline + .5*normalized_br
cross_sum_pipeline = .5*normalized_cross_pipeline + .5*normalized_br

experiment_results = pt.Experiment(
    [
        br,
        bi_pipeline,
        bi_sum_pipeline,
        cross_pipeline,
        cross_sum_pipeline
    ],
    val_queries,
    val_qrels,
    names=[
        'TF_IDF',
        'BiEnc',
        '.5*BiEnc + .5TF_IDF',
        'CrossEnc',
        '.5*CrossEnc + .5TF_IDF'
      ],
    eval_metrics=["map", 'ndcg', 'P.10']
)
display(experiment_results)

<ipython-input-27-602cc49e2e30>:4: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  br = pt.BatchRetrieve(index, wmodel='TF_IDF') % 100 # rerank only the top-100, TF_IDF, BM25


,name,map,ndcg,P.10
0,TF_IDF,0.653394,0.715161,0.082
1,BiEnc,0.805000,0.839562,0.094
2,.5*BiEnc + .5TF_IDF,0.727382,0.778359,0.088
3,CrossEnc,0.763056,0.804067,0.090
4,.5*CrossEnc + .5TF_IDF,0.690336,0.747908,0.088


In [ ]:
best_pipeline_name = experiment_results.sort_values(by=['map'], ascending=False).iloc[0]['name']

# Get the best pipeline object based on its name
best_pipeline = {
    'TF_IDF': br,
    'BiEnc': bi_pipeline,
    '.5*BiEnc + .5TF_IDF': bi_sum_pipeline,
    'CrossEnc': cross_pipeline,
    '.5*CrossEnc + .5TF_IDF': cross_sum_pipeline
}[best_pipeline_name]

# best_pipeline

**Evaluation of the best model**

In [ ]:
# evaluation of the best models on test data

pt.Experiment(
    [
        best_pipeline
    ],
    test_queries,
    test_qrels,
    names=[
        f"{best_pipeline_name}"
      ],
    eval_metrics=["map", 'ndcg', 'P.10']
)

,name,map,ndcg,P.10
0,BiEnc,0.851667,0.867869,0.09


The best model found is the Biencoder.

It is evaluated over the test set and it is performed a ranked retrieval of the 10 most relevant documents with respect to each query

In [ ]:
# Retrieve results for the 1st query
results = best_pipeline(test_queries.iloc[[0]])

# Sort results by rank in ascending order (lower rank is better)
results = results.sort_values(by=['rank'])

top_10_results = results.head(10)[['qid', 'user_id', 'docid', 'docno', 'query', 'text', 'rank']]
display(top_10_results)
# display(results)

,qid,user_id,docid,docno,query,text,rank
0,academia_185177,1532620,10928,academia_185179,after what george was georgetown university na...,georgetown university is named after the villa...,0
21,academia_185177,1532620,10800,academia_16379,after what george was georgetown university na...,here is one side effect of a university having...,1
15,academia_185177,1532620,6176,history_45209,after what george was georgetown university na...,san andreas was named after its church the fir...,2
52,academia_185177,1532620,10888,academia_69225,after what george was georgetown university na...,so focusing on the uk to start the universitie...,3
80,academia_185177,1532620,5724,judaism_98419,after what george was georgetown university na...,according to the rambam the kohen gadol pronou...,4
48,academia_185177,1532620,10889,academia_73203,after what george was georgetown university na...,applicants to a particular school may be widel...,5
96,academia_185177,1532620,8390,english_54545,after what george was georgetown university na...,behind the name says it was in use in the midd...,6
93,academia_185177,1532620,5164,judaism_3145,after what george was georgetown university na...,in kovetz ohr yisroel adar ii pp ff several pe...,7
71,academia_185177,1532620,6521,hermeneutics_520,after what george was georgetown university na...,the greek is which just means bethany where la...,8
2,academia_185177,1532620,8627,english_253024,after what george was georgetown university na...,dubs is a shorter way of saying ws the full pr...,9


In [ ]:
# Iterate the retrieval for all test queries

all_top_10_results = []  # List to store top 10 results for all queries

for i in range(test_queries.shape[0]):  # Iterate through each query
    results = best_pipeline(test_queries.iloc[[i]])  # Retrieve results for current query

    results = results.sort_values(by=['rank'])  # Sort results by rank

    top_10_results = results.head(10)[['qid', 'user_id', 'docid', 'docno', 'query', 'text', 'rank']]
    all_top_10_results.append(top_10_results)  # Append top 10 results to the list

# Concatenate all top 10 results into a single DataFrame
all_top_10_results_df = pd.concat(all_top_10_results, ignore_index=True)
best_model_retrieval_result = all_top_10_results_df.copy()

display(all_top_10_results_df)  # Display the combined results

,qid,user_id,docid,docno,query,text,rank
0,academia_185177,1532620,10928,academia_185179,after what george was georgetown university na...,georgetown university is named after the villa...,0
1,academia_185177,1532620,10800,academia_16379,after what george was georgetown university na...,here is one side effect of a university having...,1
2,academia_185177,1532620,6176,history_45209,after what george was georgetown university na...,san andreas was named after its church the fir...,2
3,academia_185177,1532620,10888,academia_69225,after what george was georgetown university na...,so focusing on the uk to start the universitie...,3
4,academia_185177,1532620,5724,judaism_98419,after what george was georgetown university na...,according to the rambam the kohen gadol pronou...,4
...,...,...,...,...,...,...,...
495,apple_425044,67390,968,sound_2610,does mac really not have keyboard shortcuts to...,hi ryani found dt very useful if you want for ...,5
496,apple_425044,67390,10370,apple_430600,does mac really not have keyboard shortcuts to...,macos has for some time now by default automat...,6
497,apple_425044,67390,169,writers_16387,does mac really not have keyboard shortcuts to...,i start my documents as nameofdocument doc or ...,7
498,apple_425044,67390,10299,apple_312822,does mac really not have keyboard shortcuts to...,el capitan introduces a feature for finding a ...,8


This section explores the use of more advanced indexing and embedding techniques for retrieval.
The corpus is indexed using PyTerrier, then embeddings of queries and documents are generated using tf-idf, cross-encoding, bi-encoding and a combination of each of the last two with tf-idf (50% each after being normalized).
MAP, NDCG, and P@10 evaluation metrics are evaluated over the query validation set; this proves the Biencoder to be the highest performing model.
Finally, this model is evaluated over the test set and the top most relevant documents according to each query are printed (for comparison).

# Query expansion

In [ ]:
# Use a pre-trained LM to expand user queries for more appropriate search results

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

torch.random.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",   # IF GPU IS USED INSTEAD OF CPU
    # device_map="cpu",    # IF CPU IS USED INSTEAD OF GPU
    torch_dtype="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

def query_expansion(query, keywords):
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": f"Given the query and the user interests given as comma separated values, provide an expanded version of the query with personalized interests. Do not add anything else. Query: {query}, Keywords: {keywords}"},
    ]

    pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    )

    generation_args = {
        "max_new_tokens": 500,
        "return_full_text": False,
        "temperature": 0.0,
        "do_sample": False,
    }

    output = pipe(messages, **generation_args)
    return output[0]['generated_text']



config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py:   0%|          | 0.00/73.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [ ]:
# Load, clean and select the first 50 queries of the dataset

test_queries = pd.read_json('answer_retrieval/test/subset_test.jsonl', lines=True)
test_queries = test_queries[['id','text','user_id', 'tags']]
test_queries.columns = ['qid', 'query', 'user_id', 'tags']
test_queries['query'] = test_queries['query'].apply(clean_text)
test_queries = test_queries.head(50)

In [ ]:
# Example usage of query expansion

first_query = test_queries.iloc[0]
original_query = first_query['query']
keywords = ", ".join(first_query['tags'])  # Convert the list of tags to a comma-separated string

expanded_query = query_expansion(original_query, keywords)

print(f"Original Query: {original_query}")
print(f"Expanded Query: {expanded_query}")

Device set to use cuda
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


Original Query: after what george was georgetown university named after what george was georgetown university named or was it named that because of where it is located
Expanded Query:  After whom was George Washington University named, and was it named after George Washington due to his association with the location, or was it named for a different reason, considering academic-history and history as key interests?


In [ ]:
# Iterate to all queries in test_queries

expanded_queries = []
for index, row in test_queries.iterrows():
    original_query = row['query']
    # Assuming 'tags' column exists. If not, you need to adjust this line accordingly.
    try:
        keywords = ", ".join(row['tags'])
    except KeyError:
        keywords = ""  # Or handle missing 'tags' differently
        print(f"Warning: 'tags' column not found for query: {original_query}")

    expanded_query = query_expansion(original_query, keywords)
    expanded_queries.append(expanded_query)

# Add the expanded queries as a new column to the test_queries dataframe
# test_queries['expanded_query'] = expanded_queries
test_queries['query'] = expanded_queries

# Now you can use the 'expanded_query' column for downstream tasks
#print(test_queries[['query', 'expanded_query']].head(5))

Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set 

In [ ]:
# Evaluate the best models on test data
# bi_pipeline is the best performing pipeline
test_queries['query'] = test_queries['query'].apply(clean_text)
pt.Experiment(
    [
        best_pipeline,
    ],
    test_queries,
    test_qrels,
    names=[
        f"{best_pipeline_name}"
      ],
    eval_metrics=["map", 'ndcg', 'P.10']
)

,name,map,ndcg,P.10
0,BiEnc,0.804583,0.827986,0.088


In [ ]:
# Retrieve results for the 1st query
results = best_pipeline(test_queries.iloc[[0]])

# Sort results by rank in ascending order (lower rank is better)
results = results.sort_values(by=['rank'])

top_10_results = results.head(10)[['qid', 'user_id', 'docid', 'docno', 'query', 'text', 'rank']]
display(top_10_results)
# display(results)

,qid,user_id,docid,docno,query,text,rank
0,academia_185177,1532620,10928,academia_185179,after whom was george washington university na...,georgetown university is named after the villa...,0
66,academia_185177,1532620,6136,history_38222,after whom was george washington university na...,no england would not have been called england ...,1
79,academia_185177,1532620,10800,academia_16379,after whom was george washington university na...,here is one side effect of a university having...,2
14,academia_185177,1532620,6176,history_45209,after whom was george washington university na...,san andreas was named after its church the fir...,3
48,academia_185177,1532620,1661,scifi_86624,after whom was george washington university na...,in the deadpool and death annual we learn the ...,4
74,academia_185177,1532620,6058,history_14308,after whom was george washington university na...,is this something that predates back many civi...,5
56,academia_185177,1532620,925,sports_26115,after whom was george washington university na...,its actually not named that way to suggest the...,6
16,academia_185177,1532620,9382,christianity_14717,after whom was george washington university na...,in addition to the various other roles that mi...,7
20,academia_185177,1532620,6628,hermeneutics_28074,after whom was george washington university na...,the data in the graphic below casts an interes...,8
65,academia_185177,1532620,4169,movies_30985,after whom was george washington university na...,according to this article i guess they just wa...,9


In [ ]:
# Iterate the retrieval for all test queries

all_top_10_results = []  # List to store top 10 results for all queries

for i in range(test_queries.shape[0]):
    results = best_pipeline(test_queries.iloc[[i]])  # Retrieve results for current query

    results = results.sort_values(by=['rank'])  # Sort results by rank

    top_10_results = results.head(10)[['qid', 'user_id', 'docid', 'docno', 'query', 'text', 'rank']]
    all_top_10_results.append(top_10_results)  # Append top 10 results to the list

# Concatenate all top 10 results into a single DataFrame
all_top_10_results_df = pd.concat(all_top_10_results, ignore_index=True)
best_model_retrieval_result_updated = all_top_10_results_df.copy()

display(all_top_10_results_df)  # Display the combined results

,qid,user_id,docid,docno,query,text,rank
0,academia_185177,1532620,10928,academia_185179,after whom was george washington university na...,georgetown university is named after the villa...,0
1,academia_185177,1532620,6136,history_38222,after whom was george washington university na...,no england would not have been called england ...,1
2,academia_185177,1532620,10800,academia_16379,after whom was george washington university na...,here is one side effect of a university having...,2
3,academia_185177,1532620,6176,history_45209,after whom was george washington university na...,san andreas was named after its church the fir...,3
4,academia_185177,1532620,1661,scifi_86624,after whom was george washington university na...,in the deadpool and death annual we learn the ...,4
...,...,...,...,...,...,...,...
495,apple_425044,67390,10263,apple_242400,query as a software developer im curious if ma...,yes they can be used interchangeably the twofi...,5
496,apple_425044,67390,10318,apple_361537,query as a software developer im curious if ma...,one way of accomplishing this is to create an ...,6
497,apple_425044,67390,10065,apple_8737,query as a software developer im curious if ma...,i guess since fnreturn equals enter the fnente...,7
498,apple_425044,67390,10105,apple_32107,query as a software developer im curious if ma...,i think not only by using a hardware keyboard ...,8


In [ ]:
display(tf_idf_retrieval_result.head(5))
display(best_model_retrieval_result.head(5))
display(best_model_retrieval_result_updated.head(5))

,qid,query,user_id,text,docno,similarity
0,academia_185177,george georgetown university named george geor...,1532620,georgetown university named village georgetown...,academia_185179,1
1,academia_185177,george georgetown university named george geor...,1532620,applicant particular school may widely dispers...,academia_73203,2
2,academia_185177,george georgetown university named george geor...,1532620,question based wrong premise funding specifica...,academia_68906,3
3,academia_185177,george georgetown university named george geor...,1532620,one side effect university famous sport team m...,academia_16379,4
4,academia_185177,george georgetown university named george geor...,1532620,yes form make appearence named dragon ball her...,anime_50580,5


,qid,user_id,docid,docno,query,text,rank
0,academia_185177,1532620,10928,academia_185179,after what george was georgetown university na...,georgetown university is named after the villa...,0
1,academia_185177,1532620,10800,academia_16379,after what george was georgetown university na...,here is one side effect of a university having...,1
2,academia_185177,1532620,6176,history_45209,after what george was georgetown university na...,san andreas was named after its church the fir...,2
3,academia_185177,1532620,10888,academia_69225,after what george was georgetown university na...,so focusing on the uk to start the universitie...,3
4,academia_185177,1532620,5724,judaism_98419,after what george was georgetown university na...,according to the rambam the kohen gadol pronou...,4


,qid,user_id,docid,docno,query,text,rank
0,academia_185177,1532620,10928,academia_185179,after whom was george washington university na...,georgetown university is named after the villa...,0
1,academia_185177,1532620,6136,history_38222,after whom was george washington university na...,no england would not have been called england ...,1
2,academia_185177,1532620,10800,academia_16379,after whom was george washington university na...,here is one side effect of a university having...,2
3,academia_185177,1532620,6176,history_45209,after whom was george washington university na...,san andreas was named after its church the fir...,3
4,academia_185177,1532620,1661,scifi_86624,after whom was george washington university na...,in the deadpool and death annual we learn the ...,4


Query expansion enriches the user's initial query by incorporating contextual features (tags in this case) that reflect their preferences and interests. It works by prompting a pretrained LM (AutoModelForCausalLM) to expand all queries and incorporate user interests.
Eventually, the extended query dataset undergoes the same retrieval and evaluation method adopted in the "More advanced IR" section, but only with the best performing pipeline (Biencoder).